# 🚀 第6周-Day6：搭建 Function Calling Agent

⚡ **今日是代码实战日！**

## 🎯 实战目标
今天我们要从零搭建一个**能调用外部工具的智能Agent**，完整体验 Function Calling 的全流程：
- 定义工具（Tool Registration）
- 构建对话循环（Agent Loop）
- 实现工具调用与结果整合
- 测试多轮工具调用场景

## 🔄 昨日复习
- Agent开发三大框架：**LangChain**（生态丰富）、**OpenAI Agents SDK**（官方简洁）、**自研轻量**（灵活可控）
- 框架选择标准：复杂度需求、延迟要求、团队技术栈
- 生产级Agent需要考虑：错误处理、重试机制、日志审计、安全边界

In [ ]:
# 环境配置
import json
import time
import random
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)
print("✅ 环境就绪")

## 🔧 架构设计：我们的 Agent 长什么样？

在动手写代码之前，先想清楚架构：

```
用户输入
    ↓
┌─────────────────┐
│   Agent Core     │  ← 对话管理器
│  (LLM + Loop)    │
└────────┬──────────┘
         ↓ LLM 判断需要工具？
    ┌────┴────┐
   是         否
    ↓          ↓
┌────────┐  直接回答
│ Tool   │  返回用户
│Registry│
└───┬────┘
    ↓
执行工具
    ↓
结果返回LLM
    ↓
生成最终回复
```

In [ ]:
# Agent 架构可视化
fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.axis('off')
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.set_title('Function Calling Agent 架构', fontsize=18, fontweight='bold', pad=20)

# 组件定义
components = {
    'user': {'pos': (7, 9), 'text': '👤 用户输入', 'color': '#E3F2FD', 'edge': '#1565C0'},
    'agent': {'pos': (7, 7), 'text': '🤖 Agent Core\nLLM + 对话循环', 'color': '#C8E6C9', 'edge': '#2E7D32'},
    'judge': {'pos': (7, 5.2), 'text': '🧐 需要工具？', 'color': '#FFF9C4', 'edge': '#F57F17'},
    'tools': {'pos': (3, 3), 'text': '🔧 Tool Registry\n工具注册表', 'color': '#BBDEFB', 'edge': '#1565C0'},
    'direct': {'pos': (11, 3), 'text': '💬 直接回答', 'color': '#F8BBD0', 'edge': '#C2185B'},
    'exec': {'pos': (3, 1), 'text': '⚙️ 执行工具\n获取结果', 'color': '#D1C4E9', 'edge': '#4527A0'},
    'reply': {'pos': (7, 0.5), 'text': '📋 生成最终回复', 'color': '#B2DFDB', 'edge': '#00695C'},
}

for name, cfg in components.items():
    x, y = cfg['pos']
    w, h = 3, 1.2
    rect = patches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                                    boxstyle="round,pad=0.15",
                                    facecolor=cfg['color'],
                                    edgecolor=cfg['edge'], linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, cfg['text'], ha='center', va='center', fontsize=11, fontweight='bold')

# 连接箭头
arrows = [
    ((7, 8.4), (7, 7.6)),    # 用户→Agent
    ((7, 6.4), (7, 5.8)),    # Agent→判断
    ((5.5, 5.2), (4.5, 3.6)),  # 判断→工具
    ((8.5, 5.2), (9.5, 3.6)),  # 判断→直接回答
    ((3, 2.4), (3, 1.6)),    # 工具→执行
    ((4.5, 1), (5.5, 0.5)),   # 执行→回复
    ((9.5, 2.4), (8.5, 0.5)),  # 直接回答→回复
]

for start, end in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# 判断分支标签
ax.text(4.2, 4.4, '是 ✅', fontsize=12, color='#2E7D32', fontweight='bold')
ax.text(9.3, 4.4, '否 ❌', fontsize=12, color='#C2185B', fontweight='bold')

plt.tight_layout()
plt.show()

## 📦 Step 1：定义工具（Tool Registration）

工具定义是 Function Calling 的基础。每个工具需要三个要素：
1. **名称**（name）：工具叫什么
2. **描述**（description）：工具做什么（LLM靠这个判断要不要调用）
3. **参数**（parameters）：工具需要什么输入（JSON Schema 格式）

我们定义5个常用工具：

In [ ]:
# Step 1: 工具定义

TOOLS = [
    {
        "name": "get_weather",
        "description": "查询指定城市的当前天气信息，包括温度、湿度、天气状况和风力",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，如'北京'、'上海'、'广州'"
                }
            },
            "required": ["city"]
        }
    },
    {
        "name": "calculate",
        "description": "执行数学计算，支持加减乘除、幂运算等基本运算",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "数学表达式，如 '2 + 3 * 4' 或 'sqrt(16)'",
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_knowledge",
        "description": "搜索知识库，查找产品信息、政策文档、常见问题解答等",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "搜索关键词或问题"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "查询指定股票的当前价格信息，包括价格、涨跌幅、成交量",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "股票代码，如'AAPL'、'TSLA'、'600519'"
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "send_notification",
        "description": "向指定用户发送通知消息，支持微信、邮件、短信等渠道",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "接收人标识（手机号、邮箱或用户ID）"
                },
                "channel": {
                    "type": "string",
                    "enum": ["wechat", "email", "sms"],
                    "description": "通知渠道"
                },
                "message": {
                    "type": "string",
                    "description": "通知内容"
                }
            },
            "required": ["to", "channel", "message"]
        }
    },
]

print("📋 已注册工具清单：")
print("=" * 50)
for i, tool in enumerate(TOOLS):
    params = ', '.join(tool['parameters']['properties'].keys())
    print(f"  {i+1}. {tool['name']}({params})")
    print(f"     描述: {tool['description'][:50]}...")
    print()

## 🛠️ Step 2：实现工具执行层

工具定义只是"说明书"，还需要真正的执行逻辑。在实际项目中，这里是调用真实API的地方。

为了方便学习，我们用**模拟数据**代替真实API调用：

In [ ]:
# Step 2: 工具执行层（模拟实现）

# 模拟天气数据
WEATHER_DATA = {
    "北京": {"temp": 32, "humidity": 45, "condition": "晴", "wind": "北风3级"},
    "上海": {"temp": 28, "humidity": 75, "condition": "多云", "wind": "东南风2级"},
    "广州": {"temp": 35, "humidity": 80, "condition": "雷阵雨", "wind": "南风4级"},
    "深圳": {"temp": 33, "humidity": 78, "condition": "多云转晴", "wind": "西南风2级"},
    "杭州": {"temp": 30, "humidity": 65, "condition": "阴", "wind": "东风1级"},
}

# 模拟知识库
KNOWLEDGE_BASE = {
    "退货": "退货流程：1.联系客服 2.提供订单号 3.等待审核（1-3工作日）4.安排退货 5.退款到原支付方式",
    "会员": "会员权益：金卡会员享9折优惠、免费配送、专属客服；银卡会员享95折优惠",
    "营业": "营业时间：周一至周五 9:00-21:00，周末及节假日 10:00-22:00",
    "支付": "支付方式：微信支付、支付宝、银行卡、现金，支持分期付款（满1000元）",
}

# 模拟股票数据
STOCK_DATA = {
    "AAPL": {"name": "苹果", "price": 195.50, "change": +2.3, "volume": "52.3M"},
    "TSLA": {"name": "特斯拉", "price": 267.80, "change": -5.1, "volume": "98.7M"},
    "600519": {"name": "贵州茅台", "price": 1688.00, "change": +1.5, "volume": "3.2M"},
}


def execute_tool(tool_name: str, arguments: dict) -> str:
    """执行工具调用，返回结果字符串"""
    print(f"  🔧 调用工具: {tool_name}({arguments})")
    
    try:
        if tool_name == "get_weather":
            city = arguments["city"]
            if city in WEATHER_DATA:
                w = WEATHER_DATA[city]
                return json.dumps({
                    "city": city,
                    "temperature": f"{w['temp']}°C",
                    "humidity": f"{w['humidity']}%",
                    "condition": w['condition'],
                    "wind": w['wind']
                }, ensure_ascii=False, indent=2)
            else:
                return f"未找到城市 '{city}' 的天气数据"

        elif tool_name == "calculate":
            expr = arguments["expression"]
            # 安全计算（仅允许数学表达式）
            allowed = set('0123456789+-*/().^ sqrt ') 
            if not all(c in allowed for c in expr):
                return "错误：表达式包含不允许的字符"
            result = eval(expr.replace('^', '**').replace('sqrt', 'np.sqrt'))
            return f"{expr} = {result}"

        elif tool_name == "search_knowledge":
            query = arguments["query"]
            results = []
            for keyword, content in KNOWLEDGE_BASE.items():
                if keyword in query or any(c in content for c in query):
                    results.append(f"[{keyword}] {content}")
            if results:
                return '\n'.join(results)
            else:
                return f"未找到与 '{query}' 相关的知识条目"

        elif tool_name == "get_stock_price":
            symbol = arguments["symbol"].upper()
            if symbol in STOCK_DATA:
                s = STOCK_DATA[symbol]
                emoji = "📈" if s["change"] >= 0 else "📉"
                return json.dumps({
                    "symbol": symbol,
                    "name": s["name"],
                    "price": f"¥{s['price']:.2f}",
                    "change": f"{emoji} {s['change']}%",
                    "volume": s["volume"]
                }, ensure_ascii=False, indent=2)
            else:
                return f"未找到股票代码 '{symbol}' 的数据"

        elif tool_name == "send_notification":
            to = arguments["to"]
            channel = arguments["channel"]
            message = arguments["message"]
            channel_names = {"wechat": "微信", "email": "邮件", "sms": "短信"}
            return f"✅ 已通过{channel_names.get(channel, channel)}向 {to} 发送通知：{message[:30]}{'...' if len(message) > 30 else ''}"

        else:
            return f"未知工具: {tool_name}"

    except Exception as e:
        return f"工具执行出错: {str(e)}"


# 测试每个工具
print("🧪 工具执行测试：")
print("=" * 50)

tests = [
    ("get_weather", {"city": "北京"}),
    ("calculate", {"expression": "(3 + 5) * 2"}),
    ("search_knowledge", {"query": "退货"}),
    ("get_stock_price", {"symbol": "AAPL"}),
    ("send_notification", {"to": "13800138000", "channel": "sms", "message": "您的订单已发货"}),
]

for tool_name, args in tests:
    print(f"\n  测试: {tool_name}")
    result = execute_tool(tool_name, args)
    print(f"  结果: {result}")
    print(f"  {'—' * 40}")

## 🧠 Step 3：构建 Agent 核心（模拟LLM决策）

在真实环境中，我们会调用 GPT/Qwen/GLM 的 API 来做工具选择决策。

这里我们用**规则模拟**来展示 Agent 的决策逻辑，帮你理解 Function Calling 的本质流程：

In [ ]:
# Step 3: Agent 核心（模拟版）

class SimpleAgent:
    """简化的 Function Calling Agent（用于教学演示）"""
    
    def __init__(self, tools: list):
        self.tools = {t['name']: t for t in tools}
        self.history = []
        self.tool_call_count = 0
    
    def _should_use_tool(self, user_message: str) -> Optional[tuple]:
        """模拟 LLM 的工具选择决策"""
        msg = user_message.lower()
        
        # 天气相关 → get_weather
        if any(w in msg for w in ['天气', '温度', '下雨', '晴', '风', '湿度']):
            city = self._extract_city(msg)
            return ('get_weather', {'city': city})
        
        # 计算相关 → calculate
        if any(w in msg for w in ['计算', '等于', '加', '减', '乘', '除', '+', '-', '*', '/', '多少']):
            expr = self._extract_expression(user_message)
            if expr:
                return ('calculate', {'expression': expr})
        
        # 知识查询 → search_knowledge
        if any(w in msg for w in ['退货', '会员', '营业', '支付', '政策', '怎么']):
            return ('search_knowledge', {'query': user_message})
        
        # 股票相关 → get_stock_price
        if any(w in msg for w in ['股票', '股价', '涨', '跌', '行情']):
            symbol = self._extract_stock_symbol(user_message)
            return ('get_stock_price', {'symbol': symbol})
        
        # 通知/发送相关 → send_notification
        if any(w in msg for w in ['通知', '发送', '提醒', '告诉']) and any(w in msg for w in ['微信', '邮件', '短信']):
            return self._extract_notification_args(user_message)
        
        return None
    
    def _extract_city(self, msg: str) -> str:
        cities = ['北京', '上海', '广州', '深圳', '杭州']
        for city in cities:
            if city in msg:
                return city
        return random.choice(cities)
    
    def _extract_expression(self, msg: str) -> Optional[str]:
        import re
        # 提取数字和运算符组成的表达式
        match = re.search(r'[\d+\-*/().^sqrt ]+', msg)
        return match.group().strip() if match else None
    
    def _extract_stock_symbol(self, msg: str) -> str:
        if 'AAPL' in msg or '苹果' in msg: return 'AAPL'
        if 'TSLA' in msg or '特斯拉' in msg: return 'TSLA'
        if '茅台' in msg or '600519' in msg: return '600519'
        return random.choice(list(STOCK_DATA.keys()))
    
    def _extract_notification_args(self, msg: str) -> tuple:
        channel = 'wechat'
        if '邮件' in msg: channel = 'email'
        elif '短信' in msg: channel = 'sms'
        return ('send_notification', {
            'to': '用户',
            'channel': channel,
            'message': msg
        })
    
    def _generate_reply(self, user_message: str, tool_result: str = None) -> str:
        """模拟 LLM 生成回复"""
        if tool_result:
            return f"根据查询结果：{tool_result}\n\n希望对你有帮助！还有其他问题吗？"
        else:
            return f"这是一个好问题！关于'{user_message}'，我目前没有可以直接查询的工具来回答，但你可以试着问我天气、计算、股票、或业务知识相关的问题。"
    
    def chat(self, user_message: str, verbose: bool = True) -> str:
        """处理用户消息的主流程"""
        self.history.append({'role': 'user', 'content': user_message})
        
        if verbose:
            print(f"\n{'='*50}")
            print(f"👤 用户: {user_message}")
        
        # Step 1: 判断是否需要工具
        tool_call = self._should_use_tool(user_message)
        
        if tool_call:
            tool_name, args = tool_call
            self.tool_call_count += 1
            
            if verbose:
                print(f"🧠 Agent决策: 需要调用工具 [{tool_name}]")
            
            # Step 2: 执行工具
            tool_result = execute_tool(tool_name, args)
            self.history.append({'role': 'tool', 'content': tool_result})
            
            # Step 3: 基于工具结果生成回复
            reply = self._generate_reply(user_message, tool_result)
        else:
            if verbose:
                print(f"🧠 Agent决策: 无需工具，直接回答")
            reply = self._generate_reply(user_message)
        
        self.history.append({'role': 'assistant', 'content': reply})
        
        if verbose:
            print(f"🤖 Agent: {reply}")
        
        return reply


# 创建 Agent 实例
agent = SimpleAgent(TOOLS)
print("✅ Agent 初始化完成！")
print(f"   已注册 {len(TOOLS)} 个工具")
print(f"   工具列表: {list(agent.tools.keys())}")

## 🧪 Step 4：测试 Agent（单轮对话）

让我们用不同的用户输入来测试 Agent 的工具选择能力：

In [ ]:
# Step 4: 单轮对话测试

test_cases = [
    "北京今天天气怎么样？",
    "帮我算一下 (128 + 256) * 3 等于多少",
    "我想了解一下退货政策",
    "查一下苹果公司的股价",
    "你好，请问你是谁？",  # 不需要工具
    "用短信通知客户13800138000，订单已发货",
]

print("🧪 单轮对话测试")
print("=" * 60)

for msg in test_cases:
    agent.chat(msg, verbose=True)

print(f"\n\n📊 统计：共调用工具 {agent.tool_call_count} 次")

## 🔄 Step 5：多轮对话与工具链

真正的 Agent 场景往往需要**多轮对话 + 多次工具调用**。

比如用户先问天气，然后根据结果决定要不要带伞。这就是 Agent 的价值所在——**上下文感知**。

In [ ]:
# Step 5: 多轮对话测试

print("🔄 多轮对话演示")
print("=" * 60)

# 场景：出行决策助手
conversation = [
    "广州今天天气怎么样？",
    "那杭州呢？",
    "帮我算一下，广州35度和杭州30度相差多少度",
    "好的，帮我发微信通知同事，杭州天气比较好，建议去杭州出差",
]

for msg in conversation:
    agent.chat(msg, verbose=True)

print(f"\n\n📊 本轮统计：共调用工具 {agent.tool_call_count} 次")
print(f"   对话历史: {len(agent.history)} 条消息")

## 📊 Step 6：可视化 Agent 运行流程

In [ ]:
# Agent 运行流程可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 左图：对话轮次 vs 工具调用
messages = agent.history
rounds = []
tool_calls = []
current_round = 0

for msg in messages:
    if msg['role'] == 'user':
        current_round += 1
    if msg['role'] == 'tool':
        rounds.append(current_round)

ax1.bar(range(1, len(rounds)+1), [1]*len(rounds), color='#42A5F5', alpha=0.8)
ax1.set_xlabel('工具调用次数')
ax1.set_ylabel('')
ax1.set_title('工具调用分布', fontsize=14, fontweight='bold')
ax1.set_xticks(range(1, len(rounds)+1))
ax1.set_xticklabels([f'第{i}次' for i in range(1, len(rounds)+1)], fontsize=9)

for i, r in enumerate(rounds):
    ax1.text(i+1, 1.05, f'轮次{r}', ha='center', fontsize=10, color='#1565C0')

# 右图：工具使用统计
tool_usage = {}
for msg in messages:
    if msg['role'] == 'tool':
        content = msg['content'][:20]
        # 从历史推断工具名
        tool_usage[content] = tool_usage.get(content, 0) + 1

colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63', '#9C27B0']
ax2.pie(
    [1] * len(TOOLS),
    labels=[t['name'] for t in TOOLS],
    colors=colors,
    autopct='%1.0f%%',
    startangle=90,
    textprops={'fontsize': 10}
)
ax2.set_title('可用工具分布', fontsize=14, fontweight='bold')

plt.suptitle('Function Calling Agent 运行分析', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🏗️ Step 7：真实项目中的 Function Calling

上面的模拟 Agent 帮你理解了原理。在真实项目中，我们会调用 LLM API 来做决策。以下是用 OpenAI API 的标准写法：

In [ ]:
# Step 7: 真实 Function Calling 代码模板（OpenAI API 格式）

REAL_AGENT_TEMPLATE = '''
import openai

client = openai.OpenAI()  # 或配置 base_url 指向国内API

def run_agent(user_message, tools, max_rounds=5):
    """带 Function Calling 的 Agent 主循环"""
    messages = [{"role": "user", "content": user_message}]
    
    for round in range(max_rounds):
        # 调用 LLM
        response = client.chat.completions.create(
            model="gpt-4o",  # 或 qwen/glm
            messages=messages,
            tools=tools,  # 传入工具定义
            tool_choice="auto"  # 让LLM自己决定
        )
        
        msg = response.choices[0].message
        
        # 情况1：LLM直接回答（不需要工具）
        if not msg.tool_calls:
            print(f"Agent: {msg.content}")
            return msg.content
        
        # 情况2：LLM要求调用工具
        messages.append(msg)  # 把LLM的工具调用请求加入历史
        
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            print(f"  调用工具: {fn_name}({fn_args})")
            
            # 执行工具
            result = execute_tool(fn_name, fn_args)
            
            # 把工具结果加入历史
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    
    return "达到最大轮次限制"
'''

print("📋 真实项目 Function Calling 模板")
print("=" * 50)
print(REAL_AGENT_TEMPLATE)

print("\n💡 关键要点：")
print("  1. tool_choice='auto' → LLM自主决定是否调用工具")
print("  2. tool_choice='required' → 强制调用工具")
print("  3. tool_choice={'name':'xxx'} → 强制调用指定工具")
print("  4. max_rounds 防止无限循环（安全措施）")

## ⚡ 关键技巧：让 Function Calling 更可靠

### 工具描述的黄金法则

LLM 依赖工具描述来决定调用哪个工具，描述写得好不好直接影响准确率。

In [ ]:
# 好的工具描述 vs 差的工具描述
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# 左图：差的工具定义
bad_tools = {
    '查询': '获取信息',
    '计算': '算数',
    '搜索': '找东西',
}

ax1.axis('off')
ax1.set_title('❌ 差的工具定义', fontsize=14, fontweight='bold', color='red')
y = 0.8
for name, desc in bad_tools.items():
    rect = patches.FancyBboxPatch((0.1, y-0.08), 0.8, 0.15,
                                    boxstyle="round,pad=0.05",
                                    facecolor='#FFEBEE', edgecolor='#EF5350')
    ax1.add_patch(rect)
    ax1.text(0.15, y, f'名称: {name}  |  描述: {desc}', fontsize=12, va='center')
    y -= 0.25

ax1.text(0.5, 0.05, '问题：描述太笼统\nLLM不知道什么时候该用哪个工具',
        ha='center', va='center', fontsize=12, color='red',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE'))

# 右图：好的工具定义
good_tools = [
    ('get_weather', '查询指定城市的实时天气，返回温度、湿度、风力等', '#C8E6C9'),
    ('calculate', '执行数学表达式计算，支持加减乘除和常用数学函数', '#C8E6C9'),
    ('search_faq', '搜索常见问题知识库，适用于退货、支付、会员等业务问题', '#C8E6C9'),
]

ax2.axis('off')
ax2.set_title('✅ 好的工具定义', fontsize=14, fontweight='bold', color='green')
y = 0.8
for name, desc, color in good_tools:
    rect = patches.FancyBboxPatch((0.05, y-0.1), 0.9, 0.18,
                                    boxstyle="round,pad=0.05",
                                    facecolor=color, edgecolor='#4CAF50')
    ax2.add_patch(rect)
    ax2.text(0.1, y+0.03, f'名称: {name}', fontsize=12, fontweight='bold', va='center')
    ax2.text(0.1, y-0.05, f'描述: {desc}', fontsize=10, va='center', color='#333')
    y -= 0.28

ax2.text(0.5, 0.05, '优点：描述具体明确\nLLM能精准判断该用哪个工具',
        ha='center', va='center', fontsize=12, color='green',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9'))

plt.suptitle('Function Calling 工具定义对比', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🛡️ 生产级 Agent 的安全要点

In [ ]:
# 生产级 Agent 安全要点
print("=" * 55)
print("🛡️  生产级 Function Calling Agent 安全检查清单")
print("=" * 55)

checklist = [
    ("参数校验", "工具执行前严格校验所有输入参数，防止注入攻击"),
    ("最大轮次", "设置 max_rounds 限制，防止 Agent 陷入无限循环"),
    ("敏感操作确认", "涉及资金、删除、发送等操作时需用户二次确认"),
    ("权限隔离", "不同用户/场景的工具权限应该隔离"),
    ("审计日志", "记录所有工具调用，便于排查和监控"),
    ("超时控制", "每个工具设置超时时间，避免阻塞"),
    ("降级策略", "工具不可用时提供优雅降级（如返回缓存数据）"),
    ("成本控制", "监控 LLM 调用 token 数，设置预算上限"),
]

for i, (item, desc) in enumerate(checklist, 1):
    print(f"\n  {i}. ✅ {item}")
    print(f"     → {desc}")

print(f"\n{'=' * 55}")
print("💡 记住：Agent 能力越强，安全边界越重要！")

## 🎓 今日总结

### 学到的核心内容

1. **Agent 架构**：用户输入 → LLM决策 → 工具调用 → 结果整合 → 最终回复
2. **工具三要素**：名称 + 描述 + 参数定义（JSON Schema）
3. **工具描述是关键**：描述越具体，LLM 选择越准确
4. **Agent Loop**：循环直到 LLM 不再需要工具（或达到最大轮次）
5. **安全边界**：参数校验、最大轮次、敏感操作确认、审计日志

### 实际开发建议
- 先用模拟数据验证流程，再接入真实 API
- 工具定义写好后先人工测试，再交给 LLM
- 生产环境必须加日志和监控
- 考虑并发场景下的工具调用冲突

In [ ]:
# 第6周学习进度
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('第6周学习进度 ⚡ Agent与工具使用', fontsize=16, fontweight='bold', pad=15)

days = [
    (1, 'Day1', 'LLM Agent基本架构', '#4CAF50', True),
    (3, 'Day2', 'Function Calling详解', '#4CAF50', True),
    (5, 'Day3', 'ReAct模式与多Agent协作', '#4CAF50', True),
    (7, 'Day4', 'Agent安全与可控性', '#4CAF50', True),
    (9, 'Day5', 'Agent开发实战框架设计', '#4CAF50', True),
    (11, 'Day6', '搭建Function Calling Agent', '#4CAF50', True),
]

for x, label, topic, color, done in days:
    circle = plt.Circle((x, 1.5), 0.6, color=color, alpha=0.3 if not done else 0.8)
    ax.add_patch(circle)
    ax.text(x, 1.7, label, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x, 1.2, topic, ha='center', va='center', fontsize=9, 
            color='#333' if done else '#999')
    if done:
        ax.text(x, 2.3, '✅', ha='center', fontsize=16)

for i in range(len(days) - 1):
    x1 = days[i][0] + 0.6
    x2 = days[i+1][0] - 0.6
    ax.plot([x1, x2], [1.5, 1.5], '-', color='#4CAF50', linewidth=2)

plt.tight_layout()
plt.show()

print("\n📊 进度：第6周/12 | Day6/7 ⚡ | 大模型 - Agent与工具使用")

## 🔑 今日英文术语

| 术语 | 音标 | 中文释义 |
|------|------|----------|
| **Tool Registration** | /tuːl ˌredʒɪˈstreɪʃən/ | 工具注册 |
| **Tool Calling** | /tuːl ˈkɔːlɪŋ/ | 工具调用 |
| **Agent Loop** | /ˈeɪdʒənt luːp/ | Agent循环（主执行流程） |
| **Function Schema** | /ˈfʌŋkʃən ˈskiːmə/ | 函数Schema（参数定义） |
| **tool_choice** | — | 工具选择策略（auto/required/none） |
| **Tool Call ID** | — | 工具调用标识（用于关联请求和结果） |
| **Max Rounds** | /mæks raʊndz/ | 最大轮次（防无限循环） |
| **Fallback** | /ˈfɔːlbæk/ | 降级策略（工具不可用时的备选方案） |
| **Guardrails** | /ˈɡɑːdreɪlz/ | 安全护栏（限制Agent行为边界） |
| **Audit Log** | /ˈɔːdɪt lɒɡ/ | 审计日志 |

## 🔄 往期回顾

**问**：ReAct 模式中 Reason 和 Act 是如何交替的？

**答**：ReAct = Reasoning + Acting 循环。模型先**推理**（"我需要做什么？应该用什么工具？"），然后**行动**（调用工具），拿到结果后再次**推理**（"结果是什么？还需要继续吗？"），如此循环直到得出最终答案。这正是今天 Agent Loop 的理论基础！

💡 明天 Day7 是复习日，我们会回顾第6周全部内容，串联 Agent 架构的知识脉络。